# 🔴 Solution: Implement Softmax（面试版）

## 📋 题目描述

**难度：** Easy

实现 softmax 函数。

Softmax 通过指数化和归一化将原始 logits 转换为概率分布，用于分类和注意力机制。

**签名:** `my_softmax(x, dim=-1) -> Tensor`

**参数:**
- `x` — 任意形状的输入张量
- `dim` — 计算 softmax 的维度

**返回:** 概率张量（沿 dim 求和为 1），形状与输入相同

**约束:**
- 在 exp 之前减去最大值以保证数值稳定
- 必须处理大值而不产生 NaN/Inf

**提示：** 1. `x_max = x.max(dim=dim, keepdim=True).values`
2. `e_x = exp(x - x_max)`
3. `return e_x / e_x.sum(dim=dim, keepdim=True)`


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    # ---- Step 1: 数值稳定性——减去最大值 ----
    # 核心技巧：softmax 对常数平移不变，减去 max 防止 exp 溢出
    # x.max(dim, keepdim) 返回 named tuple，.values 取最大值张量
    # keepdim=True：保持 dim 维度为 1，便于后续广播
    x_max = x.max(dim=dim, keepdim=True).values

    # ---- Step 2: 平移 ----
    # 平移后最大值为 0，exp(0)=1，其余 exp ∈ (0,1)
    x_shifted = x - x_max

    # ---- Step 3: 指数 ----
    # exp 是单调递增函数，保持相对大小关系
    exp_x = torch.exp(x_shifted)

    # ---- Step 4: 归一化 ----
    # sum(dim, keepdim) 沿指定维度求和，保持维度以便广播
    # 除法后每行和为 1，得到合法概率分布
    sum_exp = exp_x.sum(dim=dim, keepdim=True)
    return exp_x / sum_exp

# 面试高频考点：
# 1. 为什么要减 max？防止 exp 溢出（数值稳定性）
# 2. 为什么 keepdim=True？保持形状以便广播除法
# 3. softmax 的梯度？Jacobian 矩阵较复杂，但 CrossEntropyLoss 合并计算更高效
# 4. 温度参数？softmax(x/T)，T→0 趋近 argmax，T→∞ 趋近均匀分布

In [ ]:
# Verify
x = torch.tensor([1.0, 2.0, 3.0])
print("Output:", my_softmax(x, dim=-1))
print("Sum:   ", my_softmax(x, dim=-1).sum())
print("Ref:   ", torch.softmax(x, dim=-1))

In [ ]:
# Run judge
from torch_judge import check
check("softmax")

## 📝 核心思路总结

1. **数值稳定性**：减去 max 防止 exp 溢出，这是 softmax 实现的核心技巧
2. **keepdim 的作用**：保持求和维度，使除法能正确广播
3. **平移不变性**：softmax(x - c) = softmax(x)，数学上等价
4. **面试要点**：温度参数、梯度推导、与 CrossEntropyLoss 的关系